# Fitness Calculation from High-Throughput Functional Assay

This notebook calculates **experimental fitness scores** for RNA polymerase ribozyme variants from the high-throughput sequencing data, enabling quantitative comparison of gRNAde vs. rational design performance.

The high-throughput functional assay measures catalytic activity through selection-based enrichment:
1. **Library composition**: ~2,000 designed 5TU variants (1,000 gRNAde, 500 rational+filter, 500 rational unfiltered)
2. **In vitro selection**: Active ribozymes ligate to template and are amplified via primer extension
3. **Deep sequencing**: Pre- and post-selection libraries quantify enrichment of each variant
4. **Fitness scoring**: Log2 fold-change normalized to wild-type yields fitness scores

**Library structure:**
- **5TU variants**: 152 nt catalytic subunit with flanking sequences for PCR amplification
- **Mutational distance**: 15-40 mutations from wild-type (~80 designs per distance bin)
- **Technical replicates**: Multiple template conditions (AUA, GAA) and linker lengths (short/long)

**Selection conditions:**
- **Templates**: AUA triplet repeats (overnight), GAA triplet repeats (3h, 6h)
- **Linker variants**: Short linker (favors intramolecular reaction), long linker (allows template flexibility)
- **Controls**: Wild-type 5TU and inactive variants for calibration

## Workflow Overview

This notebook performs the following steps:
1. **Load processed sequences**: Read full-length sequences from `demultiplexed_reads_merged/rctrim/`
2. **Count variants**: Aggregate read counts for each unique sequence across all conditions
3. **Calculate fractional abundance (FA)**: Normalize counts by total library size
4. **Compute enrichment**: Calculate fold-change (post/pre) for each variant
5. **Normalize to wild-type**: Divide by wild-type enrichment to control for selection stringency
6. **Average fitness**: Mean log2 enrichment across all experimental conditions
7. **Assign to designs**: Map fitness values to design metadata CSV for downstream analysis

---

**Data sources:**
- Processed sequences: `demultiplexed_reads_merged/rctrim/*.fasta` (output from `process_raw_sequencing_data.ipynb`)
- Design metadata: `final_designs/all_designs_20032025.csv` (sequences with model, edit distance, computational scores)

In [1]:
import math
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

/home/ckj24/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Define Reference Sequences

Define wild-type 5TU sequence and experimental flanking regions used for PCR amplification:

- **Wild-type 5TU**: 152 nt catalytic subunit sequence (converted from RNA to DNA for sequencing analysis)
- **Left flank**: `AACAAACAACAAAACAAACAAACA` (24 bp, 3' primer binding site)
- **Right flank**: `GGGTCGGCATGGCATC` (16 bp, 5' primer binding site)
- **Total amplicon**: 192 bp (24 + 152 + 16)

In [2]:
# Wildtype sequence for 5TU
wt_seq = "GGAUCUUCUCGAUCUAACAAAAAAGACAAAUCUGCCACAAAGCUUGAGAGCAUCUUCGGAUGCAGAGGCGGCAGCCUUCGGUGGCGCGAUAGCGCCAACGUUCUCAACUAUGACACGCAAAACGCGUGCUCCGUUGAAUGGAGUUUAUCAUG"
wt_seq = wt_seq.replace('U', 'T')
print(len(wt_seq))

# Flanking sequences for experiment
left_seq = 'AACAAACAACAAAACAAACAAACA'
right_seq = 'GGGTCGGCATGGCATC'
print(len(left_seq + wt_seq + right_seq))

wt_seq_with_flank = left_seq + wt_seq + right_seq

152
192


# Step 1: Load and Count Sequences from Processed FASTA Files

Read full-length sequences from the processed output of `process_raw_sequencing_data.ipynb` and aggregate read counts for each unique variant.

## Load Sequence Count Data

Read processed FASTA files from `demultiplexed_reads_merged/rctrim/` and count occurrences of each unique full-length sequence (192 bp) across all experimental conditions.

**Experimental conditions loaded:**
- **Pre-selection libraries (4)**: Naïve libraries before functional selection
  - `pre_AUA_shortL`, `pre_AUA_longL`: For AUA template with short/long linker
  - `pre_GAA_shortL`, `pre_GAA_longL`: For GAA template with short/long linker

- **Post-selection libraries (6)**: Enriched libraries after functional selection
  - `post_AUA_on_shortL`, `post_AUA_on_longL`: AUA template, overnight incubation
  - `post_GAA_3h_shortL`, `post_GAA_3h_longL`: GAA template, 3-hour incubation
  - `post_GAA_6h_shortL`, `post_GAA_6h_longL`: GAA template, 6-hour incubation


In [3]:
def readFasta(fastaFile):
    fh = open(fastaFile, 'r')
    for line in fh:
        if line[0] == '>':
            header = line.rstrip()[1:]
            if sys.version_info[0] < 3:
                seq = fh.next().rstrip()
            else:
                seq = fh.readline().rstrip()
        yield [header, seq]
    fh.close()

dir_path = 'demultiplexed_reads_merged/rctrim/'
libs=[
    'pre_GAA_shortL_rc_trim_pfl',
    'post_GAA_3h_shortL_rc_trim_pfl',
    'post_GAA_6h_shortL_rc_trim_pfl',
     
    'pre_AUA_shortL_rc_trim_pfl',
    'post_AUA_on_shortL_rc_trim_pfl',
    
    'pre_GAA_longL_rc_trim_pfl',
    'post_GAA_3h_longL_rc_trim_pfl',
    'post_GAA_6h_longL_rc_trim_pfl',
    
    'pre_AUA_longL_rc_trim_pfl',
    'post_AUA_on_longL_rc_trim_pfl',
]

libraries = {}
for lib in libs:
    libraries[lib] = {}
    for line in tqdm(readFasta(dir_path + lib + '.fasta')):
        if len(line[1]) == 192:
            # read_count = int(line[0].split('-')[1])
            seq = line[1]
            if seq not in libraries[lib]:
                libraries[lib][seq] = 1
            else:
                libraries[lib][seq] += 1

37499it [00:00, 374952.81it/s]

453830it [00:00, 466732.54it/s]
508533it [00:01, 488634.19it/s]
414174it [00:00, 484253.60it/s]
399360it [00:00, 431179.32it/s]
404610it [00:00, 483989.48it/s]
493718it [00:01, 455553.15it/s]
386138it [00:00, 467885.67it/s]
395173it [00:00, 469993.21it/s]
415080it [00:00, 455701.29it/s]
530326it [00:01, 468015.49it/s]


### Summary Statistics

- **Unique sequences**: Number of distinct variants detected
- **Total reads**: Sequencing depth for each condition

High unique sequence counts and sufficient read depth ensure robust fitness estimation.

In [4]:
total_counts = {}
for lib, sub_dict in libraries.items():
    total_counts[lib] = sum(sub_dict.values())
    print(f"Library: {lib} | Unique sequences: {len(sub_dict)} | Total reads: {total_counts[lib]}")

Library: pre_GAA_shortL_rc_trim_pfl | Unique sequences: 157481 | Total reads: 453830
Library: post_GAA_3h_shortL_rc_trim_pfl | Unique sequences: 141921 | Total reads: 508533
Library: post_GAA_6h_shortL_rc_trim_pfl | Unique sequences: 139336 | Total reads: 414174
Library: pre_AUA_shortL_rc_trim_pfl | Unique sequences: 146264 | Total reads: 399360
Library: post_AUA_on_shortL_rc_trim_pfl | Unique sequences: 119129 | Total reads: 404610
Library: pre_GAA_longL_rc_trim_pfl | Unique sequences: 166375 | Total reads: 493718
Library: post_GAA_3h_longL_rc_trim_pfl | Unique sequences: 124443 | Total reads: 386138
Library: post_GAA_6h_longL_rc_trim_pfl | Unique sequences: 126758 | Total reads: 395173
Library: pre_AUA_longL_rc_trim_pfl | Unique sequences: 145583 | Total reads: 415080
Library: post_AUA_on_longL_rc_trim_pfl | Unique sequences: 131799 | Total reads: 530326


# Step 2: Calculate Fractional Abundance (FA)

Compute the fractional abundance for each sequence, defined as:

**FA = (read count for sequence) / (total reads in library)**

This normalization accounts for differences in sequencing depth across libraries, enabling fair comparison of enrichment.

## Filtering Criteria

To ensure robust fitness estimates, sequences must meet minimum count thresholds:
- **Pre-selection libraries**: ≥5 reads (sufficient starting abundance)
- **Post-selection libraries**: ≥0 reads (allows detection of complete depletion)

Sequences failing these criteria are excluded from fitness calculation to avoid noise from low-count artifacts.

## Map Post-Selection to Pre-Selection Libraries

Define the correspondence between post- and pre-selection libraries for enrichment calculation.

Each post-selection condition is paired with its corresponding pre-selection library to compute fold-change:
```
post_GAA_3h_shortL   → pre_GAA_shortL
post_GAA_6h_shortL   → pre_GAA_shortL
post_AUA_on_shortL   → pre_AUA_shortL
post_GAA_3h_longL    → pre_GAA_longL
post_GAA_6h_longL    → pre_GAA_longL
post_AUA_on_longL    → pre_AUA_longL
```

This mapping ensures correct normalization for each experimental condition.

In [5]:
# Dictionary mapping post libraries to pre libraries for different experiments
# This mapping is used for computing the fold change

post_to_pre_libs = {
    'post_GAA_3h_shortL_rc_trim_pfl': 'pre_GAA_shortL_rc_trim_pfl',
    'post_GAA_6h_shortL_rc_trim_pfl': 'pre_GAA_shortL_rc_trim_pfl',
    #
    'post_AUA_on_shortL_rc_trim_pfl': 'pre_AUA_shortL_rc_trim_pfl',
    # 
    'post_GAA_3h_longL_rc_trim_pfl': 'pre_GAA_longL_rc_trim_pfl',
    'post_GAA_6h_longL_rc_trim_pfl': 'pre_GAA_longL_rc_trim_pfl',
    # 
    'post_AUA_on_longL_rc_trim_pfl': 'pre_AUA_longL_rc_trim_pfl',
    # 
}

# only keep keys which are in 'libs'
post_to_pre_libs = {k: v for k, v in post_to_pre_libs.items() if k in libs}
print(post_to_pre_libs)

{'post_GAA_3h_shortL_rc_trim_pfl': 'pre_GAA_shortL_rc_trim_pfl', 'post_GAA_6h_shortL_rc_trim_pfl': 'pre_GAA_shortL_rc_trim_pfl', 'post_AUA_on_shortL_rc_trim_pfl': 'pre_AUA_shortL_rc_trim_pfl', 'post_GAA_3h_longL_rc_trim_pfl': 'pre_GAA_longL_rc_trim_pfl', 'post_GAA_6h_longL_rc_trim_pfl': 'pre_GAA_longL_rc_trim_pfl', 'post_AUA_on_longL_rc_trim_pfl': 'pre_AUA_longL_rc_trim_pfl'}


## Compute Fractional Abundance with Filtering

Calculate FA for all sequences passing filtering criteria:

1. **Wild-type FA**: Computed first for normalization reference
2. **All other sequences**: 
   - Check counts across all libraries
   - Apply filtering thresholds (≥5 in pre-selection, ≥0 in post-selection)
   - Calculate FA = count / total_count for passing sequences

**Output**: Dictionary `FA[sequence][library]` containing fractional abundance for each sequence in each condition.

In [6]:
# Dict to store frequency of each sequence
FA = {}

# Calculate frequency of the wildtype sequence (with flanks)
FA[wt_seq_with_flank] = {}
for lib, sub_dict in libraries.items():
    FA[wt_seq_with_flank][lib] = sub_dict.get(wt_seq_with_flank, 0) / total_counts[lib]
print(FA[wt_seq_with_flank])

# Calculate frequency of all otehr sequences
for lib in libraries:
    
    for seq in tqdm(libraries[lib].keys()):

        if seq not in FA:
            
            counts = {}
            for lib2 in libraries:
                counts[lib2] = libraries[lib2].get(seq, 0)
            
            # filtering criteria
            filter = True
            for lib2 in libraries:
                # all input libraries with at least 5 counts
                if lib2.startswith('pre') and counts[lib2] < 5:
                    filter = False
                    break
                
                # all output libraries can have 0 counts, too
                if lib2.startswith('post') and counts[lib2] < 0:
                    filter = False
                    break
            
            if filter == True:
                FA[seq] = {}
                for lib2 in libraries:
                    FA[seq][lib2] = counts[lib2] / total_counts[lib2]
                    assert FA[seq][lib2] >= 0

print("Total unique sequences with frequency data:", len(FA))

{'pre_GAA_shortL_rc_trim_pfl': 0.0003172994293017209, 'post_GAA_3h_shortL_rc_trim_pfl': 0.003287888888233409, 'post_GAA_6h_shortL_rc_trim_pfl': 0.0023709841757329044, 'pre_AUA_shortL_rc_trim_pfl': 0.0002704326923076923, 'post_AUA_on_shortL_rc_trim_pfl': 0.005672128716541855, 'pre_GAA_longL_rc_trim_pfl': 0.00034230066556212253, 'post_GAA_3h_longL_rc_trim_pfl': 0.0034573131885491716, 'post_GAA_6h_longL_rc_trim_pfl': 0.0029404842942205058, 'pre_AUA_longL_rc_trim_pfl': 0.0003035559410234172, 'post_AUA_on_longL_rc_trim_pfl': 0.008270384631339969}


100%|██████████| 131799/131799 [00:01<00:00, 129846.00it/s]

Total unique sequences with frequency data: 2005


# Step 3: Calculate Fitness Scores

Execute the fitness calculation pipeline:

### Step 1: Calculate Wild-Type Enrichment
Compute enrichment of wild-type sequence in each post-selection condition:
```python
WT_enrichment[condition] = FA_WT_post / FA_WT_pre
```

This serves as the normalization factor.

### Step 2: Calculate Variant Fitness
For each sequence:
1. **Compute enrichment ratio**: `(FA_post / FA_pre) / WT_enrichment`
2. **Convert to log2 scale**: `fitness_condition = log2(enrichment_ratio)`
   - Skip if enrichment ratio ≤ 0 (indicates severe depletion or technical error)
3. **Average across conditions**: `final_fitness = mean(fitness_all_conditions)`

### Step 3: Store Per-Condition Fitness
Maintain both:
- `fitness[seq]`: Overall fitness (mean across all conditions)
- `fitness_per_post_lib[seq][condition]`: Condition-specific fitness values

This enables analysis of condition-dependent activity differences (e.g., template preference, linker effects).

In [8]:
# Calculate the enrichment of the wildtype sequence in each library
WT_enrichment = {}
for lib in post_to_pre_libs.keys():
    WT_enrichment[lib] = FA[wt_seq_with_flank][lib] / FA[wt_seq_with_flank][post_to_pre_libs[lib]]
print(WT_enrichment)

# Calculate the fitness score for each sequence
fitness = {}
fitness_per_post_lib = {}
for seq in FA:
    
    enrichment = {}
    for lib in post_to_pre_libs.keys():
        
        # Calculate the enrichment ratio
        enrichment_ratio = (FA[seq][lib] / FA[seq][post_to_pre_libs[lib]]) / WT_enrichment[lib]
        
        # Only calculate log2 if the ratio is positive
        if enrichment_ratio > 0:
            enrichment[lib] = math.log2(enrichment_ratio)
        else:
            # Handle negative or zero ratios by setting a very negative value
            # or skipping this sequence entirely
            # enrichment[lib] = -15 
            continue
    
    if len(enrichment) > 0:
        fitness[seq] = sum(enrichment.values()) / len(enrichment)
        fitness_per_post_lib[seq] = enrichment

print("Total unique sequences with fitness data:", len(fitness))

{'post_GAA_3h_shortL_rc_trim_pfl': 10.362101487131723, 'post_GAA_6h_shortL_rc_trim_pfl': 7.472387142172667, 'post_AUA_on_shortL_rc_trim_pfl': 20.974271520723658, 'post_GAA_3h_longL_rc_trim_pfl': 10.100223389491832, 'post_GAA_6h_longL_rc_trim_pfl': 8.590355176177276, 'post_AUA_on_longL_rc_trim_pfl': 27.24500994267138}
Total unique sequences with fitness data: 1812


# Step 4: Assign Fitness to Design Metadata

Map experimental fitness values back to the design library CSV, enabling direct comparison of gRNAde vs. rational design performance.

## Fitness Assignment

For each design:
1. **Match sequence**: Use `sequence_with_flanking` to look up in fitness dictionary
2. **Assign overall fitness**: Mean fitness across all conditions
3. **Assign per-condition fitness**: Individual fitness values for each post-selection library

In [9]:
df_designs = pd.read_csv('final_designs/all_designs_20032025.csv')

# Set sequences with model == "gRNAde2_extra" to "gRNAde2"
df_designs.loc[df_designs['model'] == 'gRNAde2_extra', 'model'] = 'gRNAde2'

# Add a new column for fitness and initialize it with NaN
df_designs["fitness"] = np.nan
for i, row in df_designs.iterrows():
    seq = row['sequence_with_flanking']
    if seq in fitness:
        df_designs.at[i, 'fitness'] = fitness[seq]
    else:
        df_designs.at[i, 'fitness'] = np.nan
print("Number of sequences in designs with fitness data:", df_designs['fitness'].notna().sum())

# Add fitness per post library columns
for post_lib in post_to_pre_libs.keys():
    col_name = f"fitness_{post_lib}"
    df_designs[col_name] = np.nan
    for i, row in df_designs.iterrows():
        seq = row['sequence_with_flanking']
        if seq in fitness_per_post_lib:
            if post_lib in fitness_per_post_lib[seq]:
                df_designs.at[i, col_name] = fitness_per_post_lib[seq][post_lib]
            else:
                df_designs.at[i, col_name] = np.nan
        else:
            df_designs.at[i, col_name] = np.nan

# # remove rows with NaN fitness
# df_designs = df_designs.dropna(subset=['fitness'])
# df_designs = df_designs.reset_index(drop=True)

df_designs

Number of sequences in designs with fitness data: 1719


,fasta_desc,sequence,edit_dist,sc_score_ribonanzanet,sc_score_ribonanzanet_ss,model,seed,temperature,perplexity,sequence_with_flanking,fitness,fitness_post_GAA_3h_shortL_rc_trim_pfl,fitness_post_GAA_6h_shortL_rc_trim_pfl,fitness_post_AUA_on_shortL_rc_trim_pfl,fitness_post_GAA_3h_longL_rc_trim_pfl,fitness_post_GAA_6h_longL_rc_trim_pfl,fitness_post_AUA_on_longL_rc_trim_pfl
0,5TU,GGATCTTCTCGATCTAACAAAAAAGACAAATCTGCCACAAAGCTTG...,0,0.000000,1.000000,WT,0,0.000000,1.000000,AACAAACAACAAAACAAACAAACAGGATCTTCTCGATCTAACAAAA...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,var5TU_gRNAde0001,GGATCTTCTCGATCTATTAAAAAAGACAAATCTGCCACAAAGCTTG...,15,0.081990,0.809393,gRNAde2,3090,0.441924,1.235233,AACAAACAACAAAACAAACAAACAGGATCTTCTCGATCTATTAAAA...,-8.381630,NaN,NaN,NaN,-8.981745,-7.781515,NaN
2,var5TU_gRNAde0002,GGATCTTCTCGATCTAATCAAAAAGACAAATCTGCCACAAAGCTTG...,15,0.083147,0.830978,gRNAde2,6802,0.632620,1.212259,AACAAACAACAAAACAAACAAACAGGATCTTCTCGATCTAATCAAA...,-2.512221,-2.971837,-2.284227,-1.834079,-2.468793,-2.830984,-2.683405
3,var5TU_gRNAde0003,GGATCTTCTCGATCTAACCAACAAGACAAATCTGCCACAAAGCTTG...,15,0.089538,0.830978,gRNAde2,6802,0.632620,1.197458,AACAAACAACAAAACAAACAAACAGGATCTTCTCGATCTAACCAAC...,-2.394244,-2.523628,-2.340811,-1.868823,-2.726931,-2.883395,-2.021874
4,var5TU_gRNAde0004,GGATCTTCTCGATCCACGAAACAAGACAAATCTGCCACAATGCTTG...,15,0.091939,0.917109,gRNAde2,3809,0.743550,1.171488,AACAAACAACAAAACAAACAAACAGGATCTTCTCGATCCACGAAAC...,-0.235560,0.044039,0.170762,-0.798366,0.087741,-0.050255,-0.867283
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,var5TU_random0496,GGATATTCTCGATTCTAGCAATAAGACAAATCTGCCTCATGGCTTC...,39,0.242460,0.104234,rational_heuristic_random,4623,0.523349,0.000000,AACAAACAACAAAACAAACAAACAGGATATTCTCGATTCTAGCAAT...,-10.294976,-9.318794,NaN,NaN,NaN,NaN,-11.271157
1996,var5TU_random0497,GGACCTTCTCGGTCTGGCACTTAAGACAAATCTGCCGTAAAGCTTT...,39,0.243123,0.377730,rational_heuristic_random,1824,0.624356,0.000000,AACAAACAACAAAACAAACAAACAGGACCTTCTCGGTCTGGCACTT...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1997,var5TU_random0498,GGATTTTCTCGATCTACCAATCAAGACAAATCTGTACAAACGCTTG...,39,0.203848,0.691423,rational_heuristic_random,492,1.169611,0.000000,AACAAACAACAAAACAAACAAACAGGATTTTCTCGATCTACCAATC...,-12.198251,NaN,-11.470094,-12.543817,NaN,NaN,-12.580842
1998,var5TU_random0499,GGCTCTTCTCGAGCTAAGGATTAAGACAAATCTGCCAGAGAGCTTA...,39,0.203082,0.516794,rational_heuristic_random,3657,1.257757,0.000000,AACAAACAACAAAACAAACAAACAGGCTCTTCTCGAGCTAAGGATT...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
df_designs.to_csv('final_designs/all_designs_with_fitness_20032025.csv', index=False)